In [79]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import VOCDetection
from torchvision import models
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [80]:
S = 7
B = 2
C = 20
IMG_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

le = LabelEncoder()
le.fit(VOC_CLASSES)

img_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

In [ ]:
dataset = VOCDetection(root="./data", year="2007", image_set="train", download=True)
val_dataset = VOCDetection(root="./data", year="2007", image_set="val", download=True)

100%|██████████| 460M/460M [00:23<00:00, 19.7MB/s] 


In [ ]:
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
backbone = vgg16.features
for param in backbone.parameters():
    param.requires_grad = False

head = nn.Sequential(
    nn.Conv2d(512, 1024, kernel_size=3, padding=1),
    nn.BatchNorm2d(1024),
    nn.LeakyReLU(0.1, inplace=True),

    nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
    nn.BatchNorm2d(1024),
    nn.LeakyReLU(0.1, inplace=True),

    nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
    nn.BatchNorm2d(1024),
    nn.LeakyReLU(0.1, inplace=True),

    nn.Flatten(),
    nn.Linear(7 * 7 * 1024, 4096),
    nn.LeakyReLU(0.1, inplace=True),
    nn.Dropout(0.5),
    nn.Linear(4096, S * S * (B * 5 + C)),
)

model = nn.Sequential(backbone, head).to(DEVICE)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 141MB/s] 


In [ ]:
class YOLO:
    def __init__(self, network, lambda_coord=5, lambda_noobj=0.5):
        self.network = network
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.S = S
        self.B = B
        self.C = C

    def forward(self, x):
        out = self.network(x)
        return out.view(-1, self.S, self.S, self.B * 5 + self.C)

    def compute_iou(self, box1, box2):
        x1 = box1[..., 0] - box1[..., 2] / 2
        y1 = box1[..., 1] - box1[..., 3] / 2
        x2 = box1[..., 0] + box1[..., 2] / 2
        y2 = box1[..., 1] + box1[..., 3] / 2

        x1g = box2[..., 0] - box2[..., 2] / 2
        y1g = box2[..., 1] - box2[..., 3] / 2
        x2g = box2[..., 0] + box2[..., 2] / 2
        y2g = box2[..., 1] + box2[..., 3] / 2

        inter_x1 = torch.max(x1, x1g)
        inter_y1 = torch.max(y1, y1g)
        inter_x2 = torch.min(x2, x2g)
        inter_y2 = torch.min(y2, y2g)

        inter_w = torch.clamp(inter_x2 - inter_x1, min=0)
        inter_h = torch.clamp(inter_y2 - inter_y1, min=0)
        inter_area = inter_w * inter_h

        area1 = (x2 - x1) * (y2 - y1)
        area2 = (x2g - x1g) * (y2g - y1g)
        union = area1 + area2 - inter_area + 1e-6
        return inter_area / union

    def loss(self, y_true, y_pred):
        batch_size = y_pred.size(0)

        # Extract predictions
        p1_pred = y_pred[..., 0]
        x1_pred = y_pred[..., 1]
        y1_pred = y_pred[..., 2]
        w1_pred = F.relu(y_pred[..., 3])
        h1_pred = F.relu(y_pred[..., 4])

        p2_pred = y_pred[..., 5]
        x2_pred = y_pred[..., 6]
        y2_pred = y_pred[..., 7]
        w2_pred = F.relu(y_pred[..., 8])
        h2_pred = F.relu(y_pred[..., 9])

        cls_pred = y_pred[..., 10:]

        # Extract targets
        p1_targ = y_true[..., 0]
        x1_targ = y_true[..., 1]
        y1_targ = y_true[..., 2]
        w1_targ = y_true[..., 3]
        h1_targ = y_true[..., 4]

        p2_targ = y_true[..., 5]
        x2_targ = y_true[..., 6]
        y2_targ = y_true[..., 7]
        w2_targ = y_true[..., 8]
        h2_targ = y_true[..., 9]

        cls_targ = y_true[..., 10:]

        obj_mask = ((p1_targ + p2_targ) > 0).float().unsqueeze(-1)
        obj_mask_flat = obj_mask.squeeze(-1)

        gt_box = torch.zeros_like(y_true[..., :4])
        gt_box[..., 0] = x1_targ * p1_targ + x2_targ * p2_targ
        gt_box[..., 1] = y1_targ * p1_targ + y2_targ * p2_targ
        gt_box[..., 2] = w1_targ * p1_targ + w2_targ * p2_targ
        gt_box[..., 3] = h1_targ * p1_targ + h2_targ * p2_targ

        pred_box1 = torch.stack([x1_pred, y1_pred, w1_pred, h1_pred], dim=-1)
        pred_box2 = torch.stack([x2_pred, y2_pred, w2_pred, h2_pred], dim=-1)

        iou1 = self.compute_iou(pred_box1, gt_box)
        iou2 = self.compute_iou(pred_box2, gt_box)

        box1_resp = (iou1 >= iou2).float() * obj_mask_flat
        box2_resp = (iou2 > iou1).float() * obj_mask_flat

        noobj_mask = (1 - obj_mask_flat)

        L_box1 = box1_resp * (
            (x1_pred - x1_targ) ** 2 +
            (y1_pred - y1_targ) ** 2 +
            (torch.sqrt(w1_pred + 1e-6) - torch.sqrt(w1_targ + 1e-6)) ** 2 +
            (torch.sqrt(h1_pred + 1e-6) - torch.sqrt(h1_targ + 1e-6)) ** 2
        )
        L_box2 = box2_resp * (
            (x2_pred - x2_targ) ** 2 +
            (y2_pred - y2_targ) ** 2 +
            (torch.sqrt(w2_pred + 1e-6) - torch.sqrt(w2_targ + 1e-6)) ** 2 +
            (torch.sqrt(h2_pred + 1e-6) - torch.sqrt(h2_targ + 1e-6)) ** 2
        )
        L_box = self.lambda_coord * (L_box1.sum() + L_box2.sum())

        L_conf_obj1 = box1_resp * (p1_pred - iou1.detach()) ** 2
        L_conf_obj2 = box2_resp * (p2_pred - iou2.detach()) ** 2
        L_conf_noobj1 = self.lambda_noobj * noobj_mask * (p1_pred - 0) ** 2
        L_conf_noobj2 = self.lambda_noobj * noobj_mask * (p2_pred - 0) ** 2
        L_conf = (L_conf_obj1.sum() + L_conf_obj2.sum() +
                  L_conf_noobj1.sum() + L_conf_noobj2.sum())

        # Classification loss 
        L_cls = obj_mask * (cls_pred - cls_targ) ** 2
        L_cls = L_cls.sum()

        total_loss = (L_box + L_conf + L_cls) / batch_size
        return total_loss

    def decode(self, y_pred, conf_threshold=0.25):
        S, B, C = self.S, self.B, self.C
        p1_pred = y_pred[..., 0]
        x1_pred = y_pred[..., 1]
        y1_pred = y_pred[..., 2]
        w1_pred = F.relu(y_pred[..., 3])
        h1_pred = F.relu(y_pred[..., 4])

        p2_pred = y_pred[..., 5]
        x2_pred = y_pred[..., 6]
        y2_pred = y_pred[..., 7]
        w2_pred = F.relu(y_pred[..., 8])
        h2_pred = F.relu(y_pred[..., 9])

        cls_pred = y_pred[..., 10:]
        cls_scores, cls_labels = cls_pred.max(dim=-1)

        rows = torch.arange(S, device=y_pred.device).view(S, 1).expand(S, S)
        cols = torch.arange(S, device=y_pred.device).view(1, S).expand(S, S)

        boxes, scores, labels = [], [], []

        score1 = p1_pred * cls_scores
        mask1 = score1 > conf_threshold
        if mask1.any():
            r1 = rows[mask1].float()
            c1 = cols[mask1].float()
            bx1 = (c1 + x1_pred[mask1]) / S
            by1 = (r1 + y1_pred[mask1]) / S
            bw1 = w1_pred[mask1]
            bh1 = h1_pred[mask1]

            boxes.extend(torch.stack([bx1, by1, bw1, bh1], dim=-1).tolist())
            scores.extend(score1[mask1].tolist())
            labels.extend(cls_labels[mask1].tolist())

        score2 = p2_pred * cls_scores
        mask2 = score2 > conf_threshold
        if mask2.any():
            r2 = rows[mask2].float()
            c2 = cols[mask2].float()
            bx2 = (c2 + x2_pred[mask2]) / S
            by2 = (r2 + y2_pred[mask2]) / S
            bw2 = w2_pred[mask2]
            bh2 = h2_pred[mask2]

            boxes.extend(torch.stack([bx2, by2, bw2, bh2], dim=-1).tolist())
            scores.extend(score2[mask2].tolist())
            labels.extend(cls_labels[mask2].tolist())

        return boxes, scores, labels

    def _iou(box1, box2):
        x1 = max(box1[0] - box1[2]/2, box2[0] - box2[2]/2)
        y1 = max(box1[1] - box1[3]/2, box2[1] - box2[3]/2)
        x2 = min(box1[0] + box1[2]/2, box2[0] + box2[2]/2)
        y2 = min(box1[1] + box1[3]/2, box2[1] + box2[3]/2)
        
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = box1[2] * box1[3]
        area2 = box2[2] * box2[3]
        union = area1 + area2 - inter + 1e-6

        return inter / union

    def nms(self, boxes, scores, labels, iou_threshold=0.5):
        keep_boxes, keep_scores, keep_labels = [], [], []

        for cls in set(labels):
            idx = [i for i, l in enumerate(labels) if l == cls]
            cls_boxes = [boxes[i] for i in idx]
            cls_scores = [scores[i] for i in idx]
            order = sorted(range(len(cls_scores)), key=lambda i: cls_scores[i], reverse=True)
            while order:
                i = order.pop(0)
                keep_boxes.append(cls_boxes[i])
                keep_scores.append(cls_scores[i])
                keep_labels.append(cls)
                order = [j for j in order if self._iou(cls_boxes[i], cls_boxes[j]) < iou_threshold]

        return keep_boxes, keep_scores, keep_labels

    def predict(self, img, conf_threshold=0.25, iou_threshold=0.5):
        self.network.eval()
        img_w, img_h = img.size
        x = img_transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            y_pred = self.forward(x).squeeze(0)
        boxes, scores, labels = self.decode(y_pred, conf_threshold)
        boxes, scores, labels = self.nms(boxes, scores, labels, iou_threshold)

        detections = []
        for box, score, label in zip(boxes, scores, labels):
            xc, yc, w, h = box
            x1 = int((xc - w/2) * img_w)
            y1 = int((yc - h/2) * img_h)
            x2 = int((xc + w/2) * img_w)
            y2 = int((yc + h/2) * img_h)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_w, x2), min(img_h, y2)
            detections.append({
                "box": [x1, y1, x2, y2],
                "label": VOC_CLASSES[label],
                "score": round(score, 4)
            })
            
        return detections

    def train(self, X, Y, epochs=50, lr=1e-4, batch_size=32):
        self.network.train()
        optimizer = torch.optim.Adam(self.network.parameters(), lr=lr)

        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=10,
            gamma=0.1
        )

        N = X.shape[0]
        num_batches = (N + batch_size - 1) // batch_size
        X, Y = X.to(DEVICE), Y.to(DEVICE)

        print("Training...")
        for epoch in range(epochs):
            epoch_loss = 0.0
            perm = torch.randperm(N, device=DEVICE)
            X_shuffled, Y_shuffled = X[perm], Y[perm]
            for i in range(num_batches):
                start = i * batch_size
                end = min(start + batch_size, N)
                x_batch = X_shuffled[start:end]
                y_batch = Y_shuffled[start:end]

                y_pred = self.forward(x_batch)
                loss = self.loss(y_batch, y_pred)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.network.parameters(), max_norm=10.0)
                optimizer.step()

                epoch_loss += loss.item()
                
            scheduler.step()
            avg_loss = epoch_loss / num_batches
            print(f"Epoch {epoch+1}/{epochs} — avg loss: {avg_loss:.4f}")

In [ ]:
def parse_annotation(annotation):
    objects = annotation["annotation"]["object"]
    if not isinstance(objects, list):
        objects = [objects]
    boxes, labels = [], []
    for obj in objects:
        b = obj["bndbox"]
        boxes.append([int(b["xmin"]), int(b["ymin"]), int(b["xmax"]), int(b["ymax"])])
        labels.append(obj["name"])
    return boxes, labels

def prepare_dataset(dataset, max_cnt=None):
    x, y = [], []
    data_len = len(dataset) if max_cnt is None else max_cnt
    for i in range(data_len):
        img, annotation = dataset[i]
        img_w, img_h = img.size
        boxes, labels = parse_annotation(annotation)
        target = torch.zeros(S, S, B*5 + C)

        for box, label_name in zip(boxes, labels):
            xmin, ymin, xmax, ymax = box
            label = le.transform([label_name])[0]

            x_center = ((xmin + xmax) / 2.0) / img_w
            y_center = ((ymin + ymax) / 2.0) / img_h
            w = (xmax - xmin) / img_w
            h = (ymax - ymin) / img_h

            grid_x = min(int(x_center * S), S - 1)
            grid_y = min(int(y_center * S), S - 1)

            x_rel = x_center * S - grid_x
            y_rel = y_center * S - grid_y

            if target[grid_y, grid_x, 0] == 0:
                target[grid_y, grid_x, 0] = 1.0  # p1
                target[grid_y, grid_x, 1] = x_rel
                target[grid_y, grid_x, 2] = y_rel
                target[grid_y, grid_x, 3] = w
                target[grid_y, grid_x, 4] = h
                target[grid_y, grid_x, 10 + label] = 1.0
            elif target[grid_y, grid_x, 5] == 0:
                target[grid_y, grid_x, 5] = 1.0  # p2
                target[grid_y, grid_x, 6] = x_rel
                target[grid_y, grid_x, 7] = y_rel
                target[grid_y, grid_x, 8] = w
                target[grid_y, grid_x, 9] = h

                if target[grid_y, grid_x, 10:].sum() == 0:
                    target[grid_y, grid_x, 10 + label] = 1.0
            else:
                continue

        img_tensor = img_transform(img)
        x.append(img_tensor)
        y.append(target)
        if i % 100 == 0:
            print(f"{i+1}/{data_len}", end="\r")

    return torch.stack(x), torch.stack(y)

def visualize(img, detections):
    fig, ax = plt.subplots(figsize=(10,10))
    ax.imshow(img)
    for d in detections:
        x1, y1, x2, y2 = d["box"]
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor="red", facecolor="none"))
        ax.text(x1, y1-5, f'{d["label"]} {d["score"]:.2f}',
                color="red", fontsize=9, backgroundcolor="white")
    ax.axis("off")
    plt.show()

In [ ]:
X, Y = prepare_dataset(dataset, max_cnt=None)  

yolo = YOLO(model)
yolo.train(X, Y, epochs=70, lr=1e-4, batch_size=32)

Training...
Epoch 1/70 — avg loss: 2.0720
Epoch 2/70 — avg loss: 1.6974
Epoch 3/70 — avg loss: 1.8807
Epoch 4/70 — avg loss: 1.8296
Epoch 5/70 — avg loss: 2.1054
Epoch 6/70 — avg loss: 1.7890
Epoch 7/70 — avg loss: 1.5139
Epoch 8/70 — avg loss: 1.4890
Epoch 9/70 — avg loss: 1.4987
Epoch 10/70 — avg loss: 1.4730
Epoch 11/70 — avg loss: 1.5192
Epoch 12/70 — avg loss: 1.3589
Epoch 13/70 — avg loss: 1.4953
Epoch 14/70 — avg loss: 1.4195
Epoch 15/70 — avg loss: 1.3433
Epoch 16/70 — avg loss: 1.3652
Epoch 17/70 — avg loss: 1.3380
Epoch 18/70 — avg loss: 1.7136
Epoch 19/70 — avg loss: 1.9837
Epoch 20/70 — avg loss: 1.3098
Epoch 21/70 — avg loss: 1.2686
Epoch 22/70 — avg loss: 1.2780
Epoch 23/70 — avg loss: 1.3856
Epoch 24/70 — avg loss: 1.6443
Epoch 25/70 — avg loss: 1.3880
Epoch 26/70 — avg loss: 1.2457
Epoch 27/70 — avg loss: 1.3488
Epoch 28/70 — avg loss: 1.3513
Epoch 29/70 — avg loss: 1.1348
Epoch 30/70 — avg loss: 1.2168
Epoch 31/70 — avg loss: 1.1695
Epoch 32/70 — avg loss: 1.2112
Epoch

In [ ]:
img, annotation = val_dataset[38]
detections = yolo.predict(img)
for d in detections:
    print(d["label"], d["score"], d["box"])
visualize(img, detections)  

NameError: name 'val_dataset' is not defined